# 14 — Fine-tuning LoRA Qwen2.5-VL-3B untuk Kelas Kerusakan (Kaggle)

**Lengan eksperimen tambahan** di samping zero-shot. Pertanyaannya: *apakah SVLM yang gagal
secara zero-shot dapat membedakan kelas kerusakan setelah diadaptasi dengan sedikit data berlabel?*

| Aspek | Ketetapan |
|---|---|
| Model | Qwen2.5-VL-3B-Instruct (revisi yang sama dengan run zero-shot) |
| Metode | LoRA (r = 16, α = 32, dropout 0,05) pada seluruh lapisan linear **model bahasa**; encoder visual dibekukan |
| Tugas | kelas kerusakan dari citra **pasca** saja, dengan **prompt kelas v4 yang sama persis** dengan zero-shot |
| Target | `{"kelas_kerusakan": "<kelas>"}` (field keyakinan tidak dilatih) |
| Data latih | `sampel_latih.csv` (skrip 13): tile **terpisah** dari seluruh tile uji; ±715 latih, ±126 validasi |
| Pemilihan | 3 epoch; adapter dengan **akurasi seimbang validasi** tertinggi dipakai |
| Uji | 400 bangunan sampel besar (kelas) + 40 sampel kecil (panggilan fitur + deskripsi juga), **data uji yang sama** dengan zero-shot |
| Keluaran | run baru `runs/<RUN_ID>_qwen3b-lora_besar_...` berformat sama dengan run lain + folder `adapter/` + `pelatihan.json` |

**Prasyarat Kaggle:** GPU T4 (x2 boleh), Internet On, tiga dataset: **chip** (uji), **paket**
(prompt v4), **latih** (`kaggle_latih.zip`). Jalankan dengan **Save & Run All (Commit)**;
perkiraan total 60–90 menit. Jalankan dulu dengan `UJI_CEPAT = True` (±5 menit) untuk memastikan
loss tidak NaN, baru commit dengan `UJI_CEPAT = False`.

## Sel 1 — pustaka dan GPU

In [ ]:
!pip install -q "transformers==4.51.3" "peft==0.15.2" accelerate
import sys, torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "GPU tidak aktif"

## Sel 2 — PARAMETER (satu-satunya sel yang diubah)

`PRESISI = "fp16"` : satu GPU, bobot dasar fp16, parameter LoRA fp32, autocast + GradScaler (bawaan).
`PRESISI = "fp32"` : bobot fp32 dibagi ke dua GPU (T4 x2). Lebih lambat tetapi kebal NaN;
pakai hanya bila uji cepat fp16 memberi loss NaN.

In [ ]:
UJI_CEPAT = True        # True: 16 contoh latih, 1 epoch, 12 validasi, 6 uji. Ganti False untuk run penuh.
PRESISI = "fp16"        # "fp16" (bawaan) atau "fp32" (butuh 2 GPU)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, EPOCH, AKUMULASI, WARMUP = 1e-4, 3, 8, 20
BENIH = 42
CATATAN = "fine-tuning LoRA kelas, prompt kelas v4"
REPO = "Qwen/Qwen2.5-VL-3B-Instruct"

## Sel 3 — data

In [ ]:
import json, hashlib, importlib.util, datetime, time, random, re, gc, shutil
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

INPUT, KERJA = Path("/kaggle/input"), Path("/kaggle/working")
def cari_satu(pola):
    h = sorted(INPUT.rglob(pola))
    if not h: raise FileNotFoundError(f"{pola} tidak ada di /kaggle/input")
    return h[-1]
def sha(x, n=12):
    b = x.read_bytes() if isinstance(x, Path) else str(x).encode("utf-8")
    return hashlib.sha256(b).hexdigest()[:n]

F_KONFIG, F_PARSER = cari_satu("konfigurasi_fitur.json"), cari_satu("05_parser_keluaran.py")
F_MANIFES = cari_satu("manifes_v*_*.json")
KONFIG = json.loads(F_KONFIG.read_text(encoding="utf-8"))
MP = json.loads(F_MANIFES.read_text(encoding="utf-8"))
F_PROMPT = {j: cari_satu(MP["prompt"][j]["berkas"]) for j in ("fitur", "kelas")}
PROMPT = {j: f.read_text(encoding="utf-8") for j, f in F_PROMPT.items()}
spec = importlib.util.spec_from_file_location("parser05", F_PARSER)
P = importlib.util.module_from_spec(spec); spec.loader.exec_module(P)
PETA_KELAS = KONFIG["kelas"]["pemetaan_xbd"]            # no-damage -> tidak_rusak, dst.

F_LATIH = cari_satu("sampel_latih.csv")
latih_semua = pd.read_csv(F_LATIH)
peta_latih = {f.name: f for f in INPUT.rglob("*.png") if f.parent.name == "pasca_latih"}
latih_semua["f_pasca"] = latih_semua["chip_pasca"].map(peta_latih)
assert latih_semua["f_pasca"].notna().all(), "ada chip latih yang tidak ditemukan"
latih_semua["target"] = latih_semua["kelas_xbd"].map(PETA_KELAS)
TRAIN = latih_semua[latih_semua.peran == "latih"].reset_index(drop=True)
VAL = latih_semua[latih_semua.peran == "validasi"].reset_index(drop=True)

F_BESAR, F_KECIL = cari_satu("sampel_besar.csv"), cari_satu("sampel_kecil.csv")
UJI = pd.read_csv(F_BESAR); KECIL = set(pd.read_csv(F_KECIL)["uid"])
peta_uji = {f.name: f for f in INPUT.rglob("*.png") if f.parent.name in ("pra", "pasca")}
UJI["f_pra"], UJI["f_pasca"] = UJI["chip_pra"].map(peta_uji), UJI["chip_pasca"].map(peta_uji)
assert UJI[["f_pra", "f_pasca"]].notna().all().all(), "ada chip uji yang tidak ditemukan"
assert not set(TRAIN.tile) & set(UJI.tile), "KEBOCORAN: tile latih tumpang tindih dengan tile uji"

if UJI_CEPAT:
    TRAIN = TRAIN.groupby("target", group_keys=False).head(6).head(16)
    VAL = VAL.groupby("target", group_keys=False).head(4)
    UJI = pd.concat([UJI[UJI.uid.isin(KECIL)].groupby("kelas_xbd").head(1),
                     UJI[~UJI.uid.isin(KECIL)].groupby("kelas_xbd").head(1)])
    EPOCH = 1
print("latih  :", len(TRAIN), TRAIN.target.value_counts().to_dict())
print("validasi:", len(VAL), VAL.target.value_counts().to_dict())
print("uji    :", len(UJI), "| sampel kecil di dalamnya:", UJI.uid.isin(KECIL).sum())
print("tile latih ∩ tile uji:", len(set(TRAIN.tile) & set(UJI.tile)), "(harus 0)")

## Sel 4 — identitas run, muat model, pasang LoRA

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model

random.seed(BENIH); np.random.seed(BENIH); torch.manual_seed(BENIH)
MULAI = datetime.datetime.now(datetime.timezone.utc)
RUN_ID = (f"{MULAI:%Y%m%d-%H%M%S}_qwen3b-lora_besar{'UJICEPAT' if UJI_CEPAT else ''}"
          f"_v{MP['versi_prompt']}_{MP['sidik_fitur']}")
DIR = KERJA / "runs" / RUN_ID; DIR.mkdir(parents=True)
(DIR / "paket").mkdir()
for f in (F_KONFIG, F_PARSER, F_MANIFES, *F_PROMPT.values(), F_LATIH):
    shutil.copy(f, DIR / "paket" / f.name)
F_LOG = DIR / "log_berjalan.txt"
def catat(t):
    print(t)
    with F_LOG.open("a", encoding="utf-8") as lg: lg.write(t + "\n")

t0 = time.time()
if PRESISI == "fp32":
    assert torch.cuda.device_count() >= 2, "fp32 butuh 2 GPU (T4 x2)"
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        REPO, torch_dtype=torch.float32, device_map="auto",
        max_memory={0: "13GiB", 1: "14GiB"})
else:
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        REPO, torch_dtype=torch.float16, device_map={"": 0})
proc = AutoProcessor.from_pretrained(REPO)
try:
    from huggingface_hub import HfApi
    REVISI = HfApi().model_info(REPO).sha
except Exception:
    REVISI = getattr(base.config, "_commit_hash", None)
base.config.use_cache = False
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base.enable_input_require_grads()

# LoRA hanya pada model bahasa: model.layers.N.(self_attn|mlp).*; encoder visual (visual.*) tidak disentuh
TARGET = r"model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                 target_modules=TARGET, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(base, cfg)
for n, p_ in model.named_parameters():          # parameter latih dalam fp32 demi kestabilan
    if p_.requires_grad:
        p_.data = p_.data.float()
n_latih = sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)
n_total = sum(p_.numel() for p_ in model.parameters())
assert not any(p_.requires_grad for n, p_ in model.named_parameters() if ".visual." in n or n.startswith("visual"))
DETIK_MUAT = round(time.time() - t0, 1)
catat(f"=== {RUN_ID} | revisi {REVISI} | presisi {PRESISI} | muat {DETIK_MUAT}s")
catat(f"parameter dilatih {n_latih:,} dari {n_total:,} ({100*n_latih/n_total:.2f}%)")

## Sel 5 — penyusun contoh latih

Masukan = prompt kelas v4 + citra pasca, persis seperti panggilan kelas zero-shot.
Loss hanya dihitung pada token jawaban (token prompt dan citra diberi label -100).

In [ ]:
buka = lambda f: Image.open(f).convert("RGB")
def isi_user(gambar, teks):
    return [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
def target_teks(kelas):
    return json.dumps({"kelas_kerusakan": kelas}, ensure_ascii=False)
PERANGKAT = next(model.parameters()).device if PRESISI == "fp16" else "cuda:0"

def contoh(r):
    img = [buka(r.f_pasca)]
    pesan = [{"role": "user", "content": isi_user(img, PROMPT["kelas"])}]
    awal = proc.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)
    penuh = proc.apply_chat_template(
        pesan + [{"role": "assistant", "content": [{"type": "text", "text": target_teks(r.target)}]}],
        tokenize=False)
    x_awal = proc(text=[awal], images=img, return_tensors="pt")
    x = proc(text=[penuh], images=img, return_tensors="pt")
    n = x_awal["input_ids"].shape[1]
    assert torch.equal(x["input_ids"][0, :n], x_awal["input_ids"][0]), "awalan tidak identik"
    label = x["input_ids"].clone(); label[:, :n] = -100
    x["labels"] = label
    return {k: v.to(PERANGKAT) for k, v in x.items()}

c0 = contoh(TRAIN.iloc[0])
print("panjang urutan:", c0["input_ids"].shape[1], "| token jawaban:",
      int((c0["labels"] != -100).sum()), "->",
      proc.decode(c0["input_ids"][0][c0["labels"][0] != -100]))

## Sel 6 — fungsi evaluasi (validasi dan uji memakai generate yang sama)

In [ ]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
KELAS3 = KONFIG["kelas"]["enum"]

@torch.inference_mode()
def tanya(gambar, teks, maks):
    model.eval()
    pesan = [{"role": "user", "content": isi_user(gambar, teks)}]
    chat = proc.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)
    x = proc(text=[chat], images=gambar, return_tensors="pt", padding=True).to(PERANGKAT)
    with torch.autocast("cuda", dtype=torch.float16, enabled=(PRESISI == "fp16")):
        y = model.generate(**x, max_new_tokens=maks, do_sample=False, use_cache=True)
    return proc.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluasi_val(df):
    pred = [P.urai_kelas(tanya([buka(r.f_pasca)], PROMPT["kelas"], 24), KONFIG)["kelas_kerusakan"]
            for r in df.itertuples(index=False)]
    y = df.target.tolist(); p = [x if x in KELAS3 else "GAGAL" for x in pred]
    return {"akurasi": round(float(np.mean([a == b for a, b in zip(y, p)])), 4),
            "akurasi_seimbang": round(float(balanced_accuracy_score(y, p)), 4),
            "matriks": confusion_matrix(y, p, labels=KELAS3 + ["GAGAL"]).tolist(),
            "sebaran_prediksi": pd.Series(p).value_counts().to_dict()}

VAL0 = evaluasi_val(VAL)
catat(f"validasi SEBELUM pelatihan (zero-shot): {VAL0['akurasi_seimbang']} | {VAL0['sebaran_prediksi']}")

## Sel 7 — pelatihan

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

opt = AdamW([p_ for p_ in model.parameters() if p_.requires_grad], lr=LR, weight_decay=0.0)
langkah_total = max(1, (len(TRAIN) * EPOCH) // AKUMULASI)
sched = get_linear_schedule_with_warmup(opt, min(WARMUP, langkah_total // 5), langkah_total)
scaler = torch.amp.GradScaler("cuda", enabled=(PRESISI == "fp16"))
riwayat, terbaik = [], {"epoch": 0, "akurasi_seimbang": VAL0["akurasi_seimbang"]}
T_LATIH = time.time()
for ep in range(1, EPOCH + 1):
    model.train()
    urut = TRAIN.sample(frac=1.0, random_state=BENIH + ep).reset_index(drop=True)
    jalan, t_ep = [], time.time()
    for i, r in enumerate(urut.itertuples(index=False), 1):
        x = contoh(r)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(PRESISI == "fp16")):
            loss = model(**x).loss / AKUMULASI
        if not torch.isfinite(loss):
            raise RuntimeError(f"loss tidak hingga pada uid {r.uid}. Ulangi dengan PRESISI='fp32'.")
        scaler.scale(loss).backward()
        jalan.append(loss.item() * AKUMULASI)
        if i % AKUMULASI == 0 or i == len(urut):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p_ for p_ in model.parameters() if p_.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        if i % 20 == 0 or i == len(urut):
            catat(f"  epoch {ep} [{i:>4}/{len(urut)}] loss rerata-20 {np.mean(jalan[-20:]):.4f} | "
                  f"{(time.time()-t_ep)/i:.2f} s/contoh | uid terakhir {r.uid}")
    v = evaluasi_val(VAL)
    catat(f"epoch {ep} selesai | loss {np.mean(jalan):.4f} | validasi akurasi seimbang "
          f"{v['akurasi_seimbang']} | {v['sebaran_prediksi']}")
    riwayat.append({"epoch": ep, "loss_rerata": round(float(np.mean(jalan)), 4),
                    "detik_epoch": round(time.time() - t_ep, 1), **v})
    model.save_pretrained(DIR / f"adapter_epoch{ep}")
    if v["akurasi_seimbang"] > terbaik["akurasi_seimbang"]:
        terbaik = {"epoch": ep, "akurasi_seimbang": v["akurasi_seimbang"]}
DETIK_LATIH = round(time.time() - T_LATIH, 1)

# pakai adapter terbaik; bila tidak ada epoch yang lebih baik dari zero-shot, tetap pakai epoch terakhir
EP_PAKAI = terbaik["epoch"] or EPOCH
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
set_peft_model_state_dict(model, load_file(DIR / f"adapter_epoch{EP_PAKAI}" / "adapter_model.safetensors"))
shutil.copytree(DIR / f"adapter_epoch{EP_PAKAI}", DIR / "adapter")
for ep in range(1, EPOCH + 1):
    shutil.rmtree(DIR / f"adapter_epoch{ep}")
catat(f"pelatihan selesai {DETIK_LATIH/60:.1f} menit | adapter dipakai: epoch {EP_PAKAI}")
(DIR / "pelatihan.json").write_text(json.dumps({
    "metode": "LoRA", "r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
    "target_modules": TARGET, "encoder_visual": "dibekukan", "lr": LR, "epoch": EPOCH,
    "akumulasi_gradien": AKUMULASI, "warmup": WARMUP, "presisi": PRESISI, "benih": BENIH,
    "parameter_dilatih": n_latih, "parameter_total": n_total,
    "n_latih": len(TRAIN), "n_validasi": len(VAL), "sha_sampel_latih": sha(F_LATIH),
    "komposisi_latih": TRAIN.target.value_counts().to_dict(),
    "validasi_zero_shot": VAL0, "riwayat": riwayat, "epoch_dipakai": EP_PAKAI,
    "detik_pelatihan": DETIK_LATIH, "sha_adapter": sha(DIR / "adapter" / "adapter_model.safetensors"),
    "target_format": '{"kelas_kerusakan": "<kelas>"}', "uji_cepat": UJI_CEPAT,
}, ensure_ascii=False, indent=2), encoding="utf-8")

## Sel 8 — inferensi pada data uji (format run sama dengan zero-shot)

400 bangunan: panggilan **kelas**. 40 bangunan sampel kecil: juga panggilan **fitur + deskripsi**
(adapter tidak dilatih untuk fitur; dilaporkan apa adanya).

In [ ]:
torch.cuda.reset_peak_memory_stats()
F_MENTAH = DIR / "mentah.jsonl"
T_UJI = datetime.datetime.now(datetime.timezone.utc)
with F_MENTAH.open("w", encoding="utf-8") as fh:
    for i, r in enumerate(UJI.itertuples(index=False), 1):
        b = {"uid": r.uid, "tile": r.tile, "kelas_xbd": r.kelas_xbd, "model": "qwen2.5-vl-3b-lora",
             "run_id": RUN_ID, "mentah_fitur": None, "galat_fitur": None, "detik_fitur": None}
        t = time.time()
        b["mentah_kelas"] = tanya([buka(r.f_pasca)], PROMPT["kelas"], 64); b["galat_kelas"] = None
        b["detik_kelas"] = round(time.time() - t, 2)
        if r.uid in KECIL:
            t = time.time()
            b["mentah_fitur"] = tanya([buka(r.f_pra), buka(r.f_pasca)], PROMPT["fitur"], 256)
            b["detik_fitur"] = round(time.time() - t, 2)
        b["detik_total"] = round((b["detik_fitur"] or 0) + b["detik_kelas"], 2)
        b["selesai_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
        fh.write(json.dumps(b, ensure_ascii=False) + "\n"); fh.flush()
        hk = P.urai_kelas(b["mentah_kelas"], KONFIG)["kelas_kerusakan"]
        catat(f"[{i:>4}/{len(UJI)}] uid={r.uid} | {r.tile} | chip={r.chip_pasca} | xBD={r.kelas_xbd} | "
              f"kelas={hk}{' | +fitur' if b['mentah_fitur'] else ''} | {b['detik_total']:.1f}s")
T_UJI_SELESAI = datetime.datetime.now(datetime.timezone.utc)

## Sel 9 — urai, ringkas, manifest, arsip

In [ ]:
mentah = [json.loads(x) for x in F_MENTAH.read_text(encoding="utf-8").splitlines() if x.strip()]
rows = []
for m in mentah:
    h = {k: m.get(k) for k in ("uid", "tile", "kelas_xbd", "model", "run_id", "detik_fitur", "detik_kelas")}
    h.update(P.urai_fitur(m.get("mentah_fitur") or "", KONFIG)); h.update(P.urai_kelas(m["mentah_kelas"], KONFIG))
    rows.append(h)
df = pd.DataFrame(rows); df["label_kelas"] = df["kelas_xbd"].map(PETA_KELAS)
df.to_csv(DIR / "hasil_terurai.csv", index=False, encoding="utf-8-sig")
pred = df.kelas_kerusakan.where(df.kelas_kerusakan.isin(KELAS3), "GAGAL")
L = [f"RUN {RUN_ID}", f"model qwen2.5-vl-3b + LoRA (epoch {EP_PAKAI}) | uji n={len(df)}",
     "Matriks (baris = label xBD):\n" + pd.crosstab(df.label_kelas, pred).to_string(),
     f"akurasi {float((pred == df.label_kelas).mean()):.3f} | akurasi seimbang "
     f"{balanced_accuracy_score(df.label_kelas, pred):.3f}",
     "Riwayat validasi: " + json.dumps([(x['epoch'], x['akurasi_seimbang']) for x in riwayat]),
     f"Validasi zero-shot: {VAL0['akurasi_seimbang']}"]
dk = [m["detik_kelas"] for m in mentah]
WAKTU = {"satuan": "detik", "detik_muat_model": DETIK_MUAT, "detik_pelatihan": DETIK_LATIH,
         "per_panggilan_kelas": {"n": len(dk), "rerata": round(float(np.mean(dk)), 2),
                                 "median": round(float(np.median(dk)), 2), "sd": round(float(np.std(dk, ddof=1)), 2) if len(dk) > 1 else 0,
                                 "total_detik": round(float(np.sum(dk)), 1)},
         "jam_dinding_uji_menit": round((T_UJI_SELESAI - T_UJI).total_seconds() / 60, 1),
         "memori_gpu_puncak_uji_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
         "gpu": torch.cuda.get_device_name(0)}
(DIR / "waktu_komputasi.json").write_text(json.dumps(WAKTU, ensure_ascii=False, indent=2), encoding="utf-8")
L.append(f"Waktu: latih {DETIK_LATIH/60:.1f} menit | kelas per panggilan {WAKTU['per_panggilan_kelas']['rerata']} s")
teks = "\n".join(L); print(teks); (DIR / "ringkasan.txt").write_text(teks, encoding="utf-8")
(DIR / "manifest.json").write_text(json.dumps({
    "run_id": RUN_ID, "status": "selesai", "uji_cepat": UJI_CEPAT, "catatan": CATATAN,
    "waktu_mulai_utc": MULAI.isoformat(timespec="seconds"),
    "waktu_selesai_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "model": {"nama": "qwen2.5-vl-3b-lora", "repo": REPO, "revisi": REVISI, "dtype": PRESISI,
              "adapter": "adapter/", "sha_adapter": sha(DIR / "adapter" / "adapter_model.safetensors")},
    "lingkungan": {"platform": "kaggle", "torch": torch.__version__, "transformers": transformers.__version__,
                   "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0), "jumlah_gpu": torch.cuda.device_count()},
    "eksperimen": {"sidik_fitur": MP["sidik_fitur"], "versi_prompt": MP["versi_prompt"],
                   "fitur_terpilih": KONFIG["seleksi"]["terpilih"], "status_seleksi": KONFIG["seleksi"]["status"],
                   "sha_konfigurasi": sha(F_KONFIG), "sha_parser": sha(F_PARSER),
                   "sha_prompt_fitur": MP["prompt"]["fitur"]["sha256_12"], "sha_prompt_kelas": MP["prompt"]["kelas"]["sha256_12"],
                   "desain": "fine-tuning LoRA kelas; uji: kelas 400 + fitur 40 (sampel kecil)"},
    "sampel": {"nama": "besar", "n": int(len(UJI)), "sha_csv": sha(F_BESAR)},
    "dekode": {"do_sample": False, "max_new_tokens": {"kelas": 64, "fitur": 256}, "ukuran_citra": 448},
    "waktu_komputasi": WAKTU, "pelatihan": "pelatihan.json",
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("\narsip:", shutil.make_archive(str(KERJA / f"run_{RUN_ID}"), "zip", root_dir=KERJA / "runs", base_dir=RUN_ID))